In [4]:
# Install & clone

import os
!pip install transformers accelerate -q

REPO_DIR = "/kaggle/working/vision-search-engine"
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/3-pi-radians/vision-search-engine.git {REPO_DIR}
    %cd {REPO_DIR}

!git log --oneline -3

/kaggle/working/vision-search-engine
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.90 KiB | 1.90 MiB/s, done.
From https://github.com/3-pi-radians/vision-search-engine
   b306967..4154305  main       -> origin/main
Updating b306967..4154305
Fast-forward
 clip_encoder.py              | 95 ++++++++++++++++++++++++++++++++++++++++++++
 offline/run_clip_finetune.py |  4 +-
 2 files changed, 97 insertions(+), 2 deletions(-)
 create mode 100644 clip_encoder.py
4154305 (HEAD -> main, origin/main, origin/HEAD) Fix CLIP embedding extraction: use vision_model/text_model + projection explicitly
b306967 Commiting py file for clip fine tuning
4e376e1 fixed the path in run_blip2_caption.py -> now poins to kaggle dataset instead of working directory


In [5]:
# Verifying inputs

import os

checks = {
    "train crops":   "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/crops/train",
    "gallery crops": "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/crops/gallery",
}

for name, path in checks.items():
    exists = os.path.exists(path)
    if exists:
        count = sum(len(f) for _, _, f in os.walk(path))
        print(f"✅ {name}: {count:,} files")
    else:
        print(f"❌ {name}: NOT FOUND — attach deepfashion-inshop-crops dataset")

✅ train crops: 25,882 files
✅ gallery crops: 12,612 files


In [6]:
# Run CLIP fine-tuning

!python offline/run_clip_finetune.py

2026-04-28 12:13:02,141 INFO Device: cuda
2026-04-28 12:13:02,141 INFO Loading CLIP: openai/clip-vit-base-patch32
2026-04-28 12:13:02,239 INFO HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-28 12:13:02,259 INFO HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-04-28 12:13:02,279 INFO HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-04-28 12:13:02,280 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-28 12:13:02,307 INFO HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-04-28 12:13:02,335 INFO HTTP Request

In [7]:
# Publishing weights after training completes

import os, json

USERNAME = "pankajdeopa"
WEIGHTS_DIR = "/kaggle/working/clip_weights"

if not os.path.exists(WEIGHTS_DIR):
    print("❌ clip_weights/ not found — did Cell 3 finish?")
else:
    files = os.listdir(WEIGHTS_DIR)
    print(f"✅ clip_weights/ contains: {files}")

    meta = {
        "title": "deepfashion-clip-weights",
        "id": f"{USERNAME}/deepfashion-clip-weights",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(f"{WEIGHTS_DIR}/dataset-metadata.json", "w") as f:
        json.dump(meta, f)

    !kaggle datasets create -p {WEIGHTS_DIR} --dir-mode zip
    print(f"\n✅ Published: https://www.kaggle.com/datasets/{USERNAME}/deepfashion-clip-weights")

✅ clip_weights/ contains: ['clip_finetuned.pt']
Starting upload for file clip_finetuned.pt
100%|█████████████████████████████████████████| 577M/577M [00:04<00:00, 127MB/s]
Upload successful: clip_finetuned.pt (577MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/pankajdeopa/deepfashion-clip-weights

✅ Published: https://www.kaggle.com/datasets/pankajdeopa/deepfashion-clip-weights
